# Fine-tune the campground embedding model (T30)

Teaches the campground search model (`BAAI/bge-small-en-v1.5`, the one `dave find-campgrounds` uses) RV language like "big rig friendly" or "somewhere to plug in the RV". It trains on query/campground pairs from `dave.embed_tune` and compares recall@5 and MRR with the off-the-shelf model on held-out campgrounds and phrasings. It then exports an ONNX copy that Dave loads with `DAVE_EMBED_MODEL`.

**Before you run it:**
1. On your machine, with `RIDB_API_KEY` and `FOURSQUARE_API_KEY` set, run `uv run dave campgrounds --demo --radius 100`. More points and a bigger radius give more campgrounds; a few hundred is plenty.
2. Upload `data/campgrounds.jsonl` to Google Drive as `MyDrive/dave/campgrounds.jsonl`.

**Then:** Runtime → Change runtime type → **T4 GPU** → Run all. Training takes a few minutes.

In [ ]:
BRANCH = "main"
REPO = "https://github.com/armita-jamshidi/RV-trip-agent"
BASE_MODEL = "BAAI/bge-small-en-v1.5"
CAMPGROUNDS = "/content/drive/MyDrive/dave/campgrounds.jsonl"
OUT = "/content/drive/MyDrive/dave/campgrounds-bge-small"

## 1. Setup

In [ ]:
%pip install -q sentence-transformers==5.1.0 datasets==4.0.0 optimum[onnxruntime]==1.27.0
%pip install -q fastembed==0.9.0 lancedb pydantic
!rm -rf /content/RV-trip-agent && git clone -q --depth 1 -b $BRANCH $REPO /content/RV-trip-agent

import sys

from google.colab import drive

sys.path.insert(0, "/content/RV-trip-agent/src")
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

from datasets import Dataset

from dave import embed_tune
from dave.models import Campground

lines = Path(CAMPGROUNDS).read_text().splitlines()
camps = [Campground.model_validate_json(line) for line in lines]
train = Dataset.from_list(embed_tune.pairs(camps))
held_out = sum(embed_tune.is_test(c) for c in camps)
print(f"{len(camps)} campgrounds ({held_out} held out), {len(train)} training triples")
print(train[0])

## 2. Score the off-the-shelf model

In [ ]:
from sentence_transformers import SentenceTransformer


class STEmbedder:
    """dave's Embedder interface over a SentenceTransformer, so scoring is identical."""

    def __init__(self, model):
        self.model = model

    def embed(self, texts):
        return self.model.encode(texts, normalize_embeddings=True).tolist()

    def embed_query(self, text):
        return self.embed([text])[0]


model = SentenceTransformer(BASE_MODEL)
baseline = embed_tune.evaluate(STEmbedder(model), camps)
print(f"baseline recall@5 {baseline['recall_at_5']:.3f}, MRR {baseline['mrr']:.3f}")

## 3. Contrastive fine-tune

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

args = SentenceTransformerTrainingArguments(
    output_dir="/content/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # in-batch negatives must not repeat a query
    logging_steps=20,
    save_strategy="no",
    report_to="none",
    seed=30,
)
loss = MultipleNegativesRankingLoss(model)
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train, loss=loss)
trainer.train()
tuned = embed_tune.evaluate(STEmbedder(model), camps)
print(f"baseline recall@5 {baseline['recall_at_5']:.3f}, MRR {baseline['mrr']:.3f}")
print(f"tuned    recall@5 {tuned['recall_at_5']:.3f}, MRR {tuned['mrr']:.3f}")
beats = tuned["recall_at_5"] > baseline["recall_at_5"]
print("beats the baseline" if beats else "does NOT beat the baseline")

## 4. Export to ONNX for Dave and check the export scores the same

In [ ]:
import json
import shutil

from optimum.exporters.onnx import main_export

from dave.store.vectors import FastEmbedder

model.save_pretrained("/content/tuned")
shutil.rmtree(OUT, ignore_errors=True)
main_export("/content/tuned", output=f"{OUT}/onnx", task="feature-extraction")
for name in ["config.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"]:
    shutil.copy(f"/content/tuned/{name}", OUT)

exported = embed_tune.evaluate(FastEmbedder("/content/fastembed", Path(OUT)), camps)
print(f"ONNX export recall@5 {exported['recall_at_5']:.3f}, MRR {exported['mrr']:.3f}")
results = {"baseline": baseline, "tuned": tuned, "onnx": exported}
Path(f"{OUT}/results.json").write_text(json.dumps(results, indent=2))

## 5. Use it in Dave

Download the `campgrounds-bge-small` folder from Drive to your machine, then:

```
export DAVE_EMBED_MODEL=/path/to/campgrounds-bge-small
uv run dave eval-embeddings          # same scores as above
uv run dave index-campgrounds        # rebuild the LanceDB index with the tuned model
uv run dave find-campgrounds "big rig friendly by a lake"
```

Keep `DAVE_EMBED_MODEL` set for searching too: an index built with one model can't be searched with the other. Record the two scores in `context/decisions.md` and copy `results.json` into `training/results/embeddings.json`.